# 7. Isolation Forest
**Use case:** Financial Services — AML suspicious transaction detection for the analyst alert queue

A retail bank must monitor every transaction for money laundering and file Suspicious Activity Reports (SARs) when analysts confirm suspicion. Its legacy rule engine ('cash over $8,000') misses new patterns and produces many false alerts, so analysts waste hours while real laundering slips through. The bank has transaction amount, cash or wire type, cross-border flag, time of day, account age and each customer's history, but very few confirmed SAR labels. Success means a ranked daily alert queue where the top alerts are mostly truly suspicious, within the team's review capacity, with a reason for every alert.

> 💡 Imagine a bank teller sorting a stack of deposit slips by asking random questions like 'over $5,000?' or 'after 10 pm?'. An ordinary slip keeps landing in a crowd and needs many questions to stand alone, but a $9,800 cash deposit at 3 am from a new country is alone after one or two questions. The fewer questions needed, the more suspicious the slip.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## CONCEPT: Spot 1 weird wire among 10
See the isolation idea with random cuts by hand, then let IsolationForest flag the odd transaction among ten.

| 5W + H | |
|---|---|
| **Who** | A new AML analyst curious how the model picks alerts. |
| **What** | Ten typed-in wire transfers with amount, hour and a new-country flag; one is a 9,800 dollar wire at 3 am to a new country. |
| **When** | During analyst onboarding, before they start working the real queue. |
| **Where** | A short Python script on a laptop; the data lives in the code. |
| **Why** | To build trust that the model is not magic: odd records really do get isolated faster. |
| **How** | Write a loop that cuts randomly until one amount is alone, average over 200 runs, then fit IsolationForest and print scores and flags. |

**🧠 ML concepts:** Isolation by random splits, Path length as an anomaly signal, Averaging over many trees, contamination sets the alert threshold, Anomaly score vs label (-1 / +1)  
**🐍 Python concepts:** Tuples inside a list, while loops, Boolean mask indexing in numpy, Conditional expression (x if cond else y), enumerate and zip together, f-string alignment

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest

# 🐍 PY: a list of tuples = 10 wire transfers typed by hand: (amount_usd, hour_of_day, new_country 0/1)
txns = [
    (1200, 10, 0), (850, 11, 0), (1500, 14, 0), (990, 9, 0), (1100, 15, 0),
    (1300, 13, 0), (700, 16, 0), (1250, 10, 0), (9800, 3, 1), (1050, 12, 0),
]
# 🐍 PY: np.array turns the list into a 10x3 numeric grid that sklearn can read
X = np.array(txns)
print("Shape of data:", X.shape)

# 🧠 ML: core idea - an odd point gets isolated with FEW random cuts; a normal one needs many
rng = np.random.default_rng(42)
amounts = X[:, 0]
# 🐍 PY: a function that keeps cutting at random until only the target value is left
def cuts_to_isolate(values, target, rng):
    cuts = 0
    while len(values) > 1:
        # 🧠 ML: pick a random split point between the current min and max
        split = rng.uniform(values.min(), values.max())
        # 🐍 PY: boolean mask keeps only the side of the split that holds our target
        values = values[values < split] if target < split else values[values >= split]
        cuts += 1
    return cuts

# 🧠 ML: average over many random trees, exactly what a forest does
for idx in [0, 8]:
    # 🐍 PY: list comprehension repeats the experiment 200 times
    runs = [cuts_to_isolate(amounts, amounts[idx], rng) for _ in range(200)]
    print(f"txn {idx} amount={amounts[idx]:>6.0f}: avg cuts to isolate = {np.mean(runs):.2f}")

# 🧠 ML: contamination=0.1 tells the model we expect about 1 in 10 to be suspicious
model = IsolationForest(n_estimators=100, contamination=0.1, random_state=42)
model.fit(X)
# 🧠 ML: predict returns -1 for anomaly, +1 for normal; score_samples: lower = more unusual
labels = model.predict(X)
scores = model.score_samples(X)

print("\n  #  amount hour newCtry  score  flag")
# 🐍 PY: enumerate gives the row number; zip walks three sequences side by side
for i, (row, s, lab) in enumerate(zip(txns, scores, labels)):
    flag = "ALERT" if lab == -1 else ""
    print(f"{i:3d} {row[0]:7d} {row[1]:4d} {row[2]:7d} {s:6.3f}  {flag}")

**Try this next:**
- Change transaction 8 to 1,400 dollars at hour 14 with newCtry 0 and see if anything is still flagged.
- Set contamination=0.2 and notice a second, borderline wire becomes an alert.
- Add a second suspicious wire and check whether two anomalies 'hide' each other.

## BUILD: Score 5,000 transactions into an analyst queue
Fit IsolationForest on realistic synthetic transactions and judge it the way an AML team does: by precision in the top of the queue.

| 5W + H | |
|---|---|
| **Who** | The AML analytics team evaluating a model to supplement legacy rules. |
| **What** | 5,000 synthetic transactions (2 percent suspicious) with amount, night, cash, cross-border, velocity and account age. |
| **When** | During a proof of concept, using a labelled sample of past SAR outcomes to grade the unsupervised model. |
| **Where** | An analytics notebook against a transaction extract. |
| **Why** | Analysts can only review so many alerts, so the top of the ranked list must be rich in true suspicious activity. |
| **How** | Log-transform amount, fit IsolationForest on features only, compute ROC-AUC, precision and recall, and precision at top-10/30/50. |

**🧠 ML concepts:** Unsupervised training, labelled evaluation, Log transform for skewed amounts, ROC-AUC for ranking quality, Precision and recall of alerts, Precision at K for capacity-limited queues  
**🐍 Python concepts:** Helper function returning a DataFrame, pd.concat and sample(frac=1) to shuffle, Boolean to int with astype, nlargest to rank rows, groupby with mean for profiles

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score

np.random.seed(42)
n_normal, n_bad = 4900, 100

# 🐍 PY: helper that builds one block of transactions so we can reuse it for normal and suspicious rows
def make_txns(n, amt_mu, amt_sd, night_p, cash_p, xborder_p, velocity_lam, label):
    return pd.DataFrame({
        "txn_amount": np.abs(np.random.lognormal(amt_mu, amt_sd, n)).round(2),
        "is_night": np.random.binomial(1, night_p, n),
        "cash_deposit": np.random.binomial(1, cash_p, n),
        "cross_border": np.random.binomial(1, xborder_p, n),
        "txns_last_24h": np.random.poisson(velocity_lam, n),
        "acct_age_days": np.random.randint(30, 4000, n) if label == 0 else np.random.randint(5, 900, n),
        "is_suspicious": label,
    })

# 🧠 ML: suspicious = larger amounts (thousands), more cash, more cross-border, high velocity, new accounts
normal = make_txns(n_normal, 6.0, 1.0, 0.05, 0.10, 0.03, 2, 0)
bad = make_txns(n_bad, 8.6, 0.35, 0.25, 0.45, 0.25, 5, 1)
# 🐍 PY: pd.concat stacks DataFrames; sample(frac=1) shuffles all rows
df = pd.concat([normal, bad]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Rows: {len(df)}  suspicious rate: {df.is_suspicious.mean():.1%}")

# 🧠 ML: log of amount tames the long tail so one $500k wire does not dominate the cuts
df["log_amount"] = np.log1p(df.txn_amount)
feats = ["log_amount", "is_night", "cash_deposit", "cross_border", "txns_last_24h", "acct_age_days"]
train, test = train_test_split(df, test_size=0.3, random_state=42, stratify=df.is_suspicious)

# 🧠 ML: unsupervised - we fit on features only; labels are kept aside just to grade the result
model = IsolationForest(n_estimators=200, contamination=0.02, random_state=42)
model.fit(train[feats])

# 🧠 ML: negate score_samples so that a HIGHER number = MORE suspicious (easier for analysts)
test = test.copy()
test["risk_score"] = -model.score_samples(test[feats])
# 🐍 PY: (predict == -1) gives True/False; astype(int) turns it into 1/0
test["alert"] = (model.predict(test[feats]) == -1).astype(int)

auc = roc_auc_score(test.is_suspicious, test.risk_score)
p = precision_score(test.is_suspicious, test.alert)
r = recall_score(test.is_suspicious, test.alert)
print(f"ROC-AUC: {auc:.3f}   alert precision: {p:.2f}   alert recall: {r:.2f}")
print(f"Alerts raised: {test.alert.sum()} of {len(test)} test transactions")

# 🧠 ML: analysts work a ranked queue, so precision in the top-K matters more than overall accuracy
for k in [10, 30, 50]:
    # 🐍 PY: nlargest(k, col) returns the k rows with the highest risk score
    topk = test.nlargest(k, "risk_score")
    print(f"  top-{k:<3d} queue: {topk.is_suspicious.sum():2d} truly suspicious ({topk.is_suspicious.mean():.0%})")

print("\nTop 5 alerts for the AML analyst:")
cols = ["txn_amount", "cash_deposit", "cross_border", "txns_last_24h", "acct_age_days", "risk_score"]
print(test.nlargest(5, "risk_score")[cols].round(3).to_string(index=False))

# 🐍 PY: groupby + mean compares the average profile of alerted vs non-alerted transactions
print("\nAverage profile by alert flag:")
print(test.groupby("alert")[["txn_amount", "cash_deposit", "txns_last_24h"]].mean().round(2))

**Try this next:**
- Drop log_amount and use raw txn_amount; compare ROC-AUC.
- Change contamination to 0.01 and 0.05 and watch precision and recall trade off.
- Remove acct_age_days from feats and see which top alerts change.

## PRODUCTION: Customer-aware AML scoring with tuning, baseline and reason codes
Build behaviour-based AML features, tune IsolationForest on PR-AUC with a time-ordered split, beat a legacy rule on cost and explain each alert.

| 5W + H | |
|---|---|
| **Who** | The AML model owner, the BSA officer and model validation (second line). |
| **What** | 60 days of daily activity for 600 customers, a few of whom start laundering; features compare each transaction to the customer's own history. |
| **When** | Before replacing or supplementing the legacy rule in the production transaction-monitoring system. |
| **Where** | The bank's model development platform; the saved model is deployed to the nightly scoring batch. |
| **Why** | Regulators expect a risk-based, tuned and validated monitoring program; analysts need fewer false alerts and clear reasons. |
| **How** | Per-customer features (amount vs own median, 3-day velocity, near-10k flag), time-ordered train/test, grid over trees, sample size and feature fraction scored by PR-AUC, top-K capacity analysis, cost vs rule, median-reset explanations and joblib save. |

**🧠 ML concepts:** Behavioural features relative to a customer's own history, Time-ordered split to avoid look-ahead, PR-AUC for rare positives, Capacity-based thresholds (top-K), Local explanations by feature reset, Model risk under SR 11-7  
**🐍 Python concepts:** Nested loops for a grid search, groupby().transform with lambda, np.searchsorted for rolling counts, sorted with a key lambda, Pipeline with FunctionTransformer, Docstrings and helper functions

In [ ]:
import os
import tempfile
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore")
SEED = 42
FEATS = ["log_amount", "amt_vs_cust_avg", "is_night", "cash_deposit", "cross_border",
         "txns_last_3d", "near_10k", "acct_age_days"]


def simulate(n_cust=600, days=60, bad_cust=12, seed=SEED):
    """Daily transactions per customer; a few customers start laundering in the last 20 days."""
    rng = np.random.default_rng(seed)
    rows = []
    # 🐍 PY: a set gives fast 'c in bad_ids' membership checks inside the loop
    bad_ids = set(rng.choice(n_cust, bad_cust, replace=False))
    for c in range(n_cust):
        base = rng.lognormal(6, 0.8)
        age = rng.integers(30, 4000)
        for d in range(days):
            # 🧠 ML: laundering customers transact more often in their final 20 days (velocity)
            active_p = 0.5 if (c in bad_ids and d >= days - 20) else 0.35
            if rng.random() > active_p:
                continue
            # 🧠 ML: labels exist only to GRADE the model; Isolation Forest never sees them
            launder = c in bad_ids and d >= days - 20 and rng.random() < 0.6
            amt = rng.uniform(2500, 9990) if launder else base * rng.lognormal(0, 0.5)
            rows.append(dict(cust_id=c, day=d, txn_amount=round(amt, 2), acct_age_days=age + d,
                             is_night=int(rng.random() < (0.25 if launder else 0.08)),
                             cash_deposit=int(rng.random() < (0.5 if launder else 0.15)),
                             cross_border=int(rng.random() < (0.2 if launder else 0.05)),
                             is_suspicious=int(launder)))
    return pd.DataFrame(rows)


def add_features(df):
    df = df.sort_values(["cust_id", "day"]).copy()
    # 🧠 ML: log1p compresses huge amounts so random cuts are not wasted on the long tail
    df["log_amount"] = np.log1p(df.txn_amount)
    # 🧠 ML: behaviour vs the customer's OWN history catches anomalies a global rule would miss
    # 🐍 PY: shift() moves values down one row so today's amount is excluded from its own history
    hist = df.groupby("cust_id").txn_amount.transform(lambda s: s.shift().expanding().median())
    df["amt_vs_cust_avg"] = (df.txn_amount / hist.fillna(df.txn_amount)).clip(0, 50)
    # 🧠 ML: velocity feature = how many txns this customer made in the last few days
    # 🐍 PY: searchsorted on the sorted day array counts rows with day in [d-2, d] for each row
    df["txns_last_3d"] = df.groupby("cust_id").day.transform(
        lambda s: np.arange(len(s)) - np.searchsorted(s.values, s.values - 2) + 1)
    # 🧠 ML: structuring red flag - amounts just below the $10k CTR reporting threshold
    df["near_10k"] = df.txn_amount.between(8000, 9999).astype(int)
    return df


# 🐍 PY: np.argsort(-score) sorts highest-risk first; [:k] keeps the analyst's daily capacity
def queue_metrics(y, score, k):
    order = np.argsort(-score)[:k]
    return y[order].sum(), y.sum()


df = add_features(simulate())
# 🧠 ML: time-ordered split - fit on days 0-39 (mostly clean history), score the newest 20 days
train, test = df[df.day < 40], df[df.day >= 40]
print(f"Train txns={len(train)}  Test txns={len(test)}  test suspicious={test.is_suspicious.sum()}")

# 🧠 ML: baseline = classic AML rule 'cash deposit over $8k' used by many legacy rule engines
rule = ((test.cash_deposit == 1) & (test.txn_amount > 8000)).astype(int).values
y = test.is_suspicious.values

# 🧠 ML: tune n_estimators / max_samples / max_features; score with PR-AUC because positives are rare
best, best_ap = None, -1
for n_est in [100, 300]:
    for max_s in [256, 1024]:
        for max_f in [0.6, 1.0]:
            # 🧠 ML: max_features < 1 lets each tree see a random subset of features (more diversity)
            m = Pipeline([("sel", FunctionTransformer(lambda d: d[FEATS].values)),
                          ("iso", IsolationForest(n_estimators=n_est, max_samples=max_s,
                                                  max_features=max_f, random_state=SEED))])
            m.fit(train)
            ap = average_precision_score(y, -m.score_samples(test))
            if ap > best_ap:
                best, best_ap, best_p = m, ap, (n_est, max_s, max_f)
print(f"Best (n_estimators, max_samples, max_features) = {best_p}  PR-AUC={best_ap:.3f}")
# Note: in production you tune on a separate validation window, not the final test window.

# 🧠 ML: negate so a higher number means more suspicious; ROC-AUC checks overall ranking quality
score = -best.score_samples(test)
print(f"ROC-AUC={roc_auc_score(y, score):.3f}")

# 🧠 ML: business threshold = analyst capacity; 25 analyst-hours/day at ~1 alert/hour
for k in [25, 50, 100]:
    hit, tot = queue_metrics(y, score, k)
    print(f"  top-{k:<3d} alerts -> {hit:3d}/{tot} suspicious caught  precision={hit / k:.0%}")
# 🐍 PY: & on two 0/1 arrays is element-wise AND - counts alerts that were truly suspicious
rule_hits = (rule & y).sum()
print(f"  rule baseline   -> {rule_hits:3d}/{y.sum()} caught using {rule.sum()} alerts"
      f"  precision={rule_hits / max(rule.sum(), 1):.0%}")

# 🧠 ML: explain one alert - how much the score drops when each feature is reset to the median
# 🧠 ML: reset one feature at a time to the 'typical' median; the bigger the score drop, the bigger the reason
top_idx = int(np.argmax(score))
row = test.iloc[[top_idx]].copy()
med = train[FEATS].median()
contrib = {}
for f in FEATS:
    r2 = row.copy()
    r2[f] = med[f]
    contrib[f] = score[top_idx] - (-best.score_samples(r2)[0])
print(f"\nWhy is cust {row.cust_id.iat[0]} day {row.day.iat[0]} top of queue? (score={score[top_idx]:.3f})")
for f, v in sorted(contrib.items(), key=lambda kv: -kv[1])[:5]:
    print(f"  {f:16s} value={row[f].iat[0]:>9.2f}  score lift={v:+.3f}")

# 🧠 ML: cost view - missed SAR is far costlier than an extra analyst review
cost_review, cost_miss = 40, 5000
hit, tot = queue_metrics(y, score, 100)
print(f"\nCost @top-100: model=${100 * cost_review + (tot - hit) * cost_miss:,}"
      f"  rule=${rule.sum() * cost_review + (tot - rule_hits) * cost_miss:,}")

# Model-risk: BSA/AML models fall under SR 11-7 validation; FinCEN expects documented tuning,
# below-the-line testing, and analyst feedback. Scores route to humans; a person decides on SAR filing.
print("Governance: SR 11-7 validation + BSA/AML tuning docs; analyst, not model, files the SAR.")
# 🧠 ML: persist the fitted forest so nightly scoring uses the exact validated model
path = os.path.join(tempfile.mkdtemp(), "aml_iforest.joblib")
joblib.dump(best.named_steps["iso"], path)
print("Saved ->", os.path.basename(path))

**Try this next:**
- Lower the laundering amount range to 500-3,000 dollars and see whether amt_vs_cust_avg still finds them.
- Add a per-customer cross-border share feature and re-run the grid.
- Replace top-K with a score threshold chosen on days 30-39 and report precision on days 40-59.